# 01 - Data Conditioning and Pre-processing

This notebook implements a reproducible extraction and preprocessing path from polymarket_events.csv into a microstructure-like time series.

Core formulas:

1. Weighted canonical mid:
$$
	ilde p_t = rac{um_{u n (t-elta,t]} w_u dot m_u}{um_{u n (t-elta,t]} w_u}, uad m_u = rac{b_u+a_u}{2}, uad w_u = rac{qrt{q_u}}{ax(a_u-b_u,psilon)}
$$
2. Boundary clipping:
$$
p_t = in(1-arepsilon,ax(arepsilon,	ilde p_t)), uad arepsilon=10^{-5}
$$
3. Uniform-grid resampling:
$$
p_{t_k} = 	ext{LOCF/VWAP over }[t_{k-1}, t_k]
$$

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path

## 1) Load raw polymarket events

In [3]:
raw_path = Path('../polymarket_events.csv')
if not raw_path.exists():
    raw_path = Path('polymarket_events.csv')

raw = pd.read_csv(raw_path)
print(f'Rows: {len(raw):,} | Cols: {len(raw.columns)}')
raw.columns[:20].tolist()

Rows: 43,840 | Cols: 80


C:\Users\p\AppData\Local\Temp\ipykernel_3740\2355908178.py:5: DtypeWarning: Columns (0: automaticallyActive, 1: cantEstimate, 2: carouselMap, 3: category, 4: commentsEnabled, 5: countryName, 6: disqusThread, 7: elapsed, 8: electionType, 9: enableOrderBook, 10: estimateValue, 11: eventCreators, 12: featured, 13: featuredImage, 14: new, 15: published_at, 16: subcategory) have mixed types. Specify dtype option on import or set low_memory=False.
  raw = pd.read_csv(raw_path)


['active',
 'archived',
 'automaticallyActive',
 'automaticallyResolved',
 'awayTeamName',
 'cantEstimate',
 'carouselMap',
 'category',
 'closed',
 'closedTime',
 'commentCount',
 'commentsEnabled',
 'competitive',
 'countryName',
 'createdAt',
 'createdBy',
 'creationDate',
 'cyom',
 'deploying',
 'deployingTimestamp']

## 2) Extract bid/ask/trade-like fields from polymarket_events

Notes:
- If explicit bid/ask/trade columns exist, they are used.
- If they do not exist (event-level snapshot data), a documented proxy is constructed from liquidity/volume fields so downstream steps remain executable.

In [4]:
def pick_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

ts_col = pick_col(raw, ['updatedAt', 'createdAt', 'startDate', 'eventDate'])
id_col = pick_col(raw, ['id', 'slug', 'ticker'])

mid_col = pick_col(raw, ['estimateValue', 'estimatedValue', 'mid', 'price'])
bid_col = pick_col(raw, ['bid', 'bestBid', 'bidPrice'])
ask_col = pick_col(raw, ['ask', 'bestAsk', 'askPrice'])
trade_col = pick_col(raw, ['trade_size', 'size', 'volume24hr', 'volume'])
liq_col = pick_col(raw, ['liquidityClob', 'liquidityAmm', 'liquidity'])

work = raw.copy()
work['timestamp'] = pd.to_datetime(work[ts_col], utc=True, errors='coerce')
work['event_id'] = work[id_col].astype(str)

# Mid probability extraction
if mid_col is not None:
    mid_raw = pd.to_numeric(work[mid_col], errors='coerce')
else:
    fallback = pd.to_numeric(work.get('openInterest', np.nan), errors='coerce')
    mid_raw = (fallback / (fallback.max() + 1e-12)).clip(0.01, 0.99)

# Bid/ask extraction or proxy spread construction
if bid_col is not None and ask_col is not None:
    bid = pd.to_numeric(work[bid_col], errors='coerce')
    ask = pd.to_numeric(work[ask_col], errors='coerce')
else:
    liq = pd.to_numeric(work[liq_col], errors='coerce') if liq_col else pd.Series(np.nan, index=work.index)
    liq = liq.fillna(liq.median() if np.isfinite(liq).any() else 1000.0)
    spread = (0.01 + 2000.0 / (liq + 2000.0) * 0.08).clip(0.005, 0.12)
    bid = (mid_raw - 0.5 * spread).clip(1e-5, 1 - 1e-5)
    ask = (mid_raw + 0.5 * spread).clip(1e-5, 1 - 1e-5)

# Trade-size extraction or proxy
if trade_col is not None:
    trade_size = pd.to_numeric(work[trade_col], errors='coerce').abs().fillna(0.0)
else:
    vol = pd.to_numeric(work.get('volume24hr', np.nan), errors='coerce').fillna(0.0)
    trade_size = vol.groupby(work['event_id']).diff().abs().fillna(0.0)

micro = pd.DataFrame({
    'timestamp': work['timestamp'],
    'event_id': work['event_id'],
    'mid_raw': mid_raw,
    'bid': bid,
    'ask': ask,
    'trade_size': trade_size,
})
micro = micro.dropna(subset=['timestamp', 'mid_raw', 'bid', 'ask']).sort_values(['event_id', 'timestamp'])
micro.head()

,timestamp,event_id,mid_raw,bid,ask,trade_size
40513,2025-11-14 16:31:16.525802+00:00,61075,1.0,0.955,0.99999,0.0


## 3) Canonical mid, clipping, and resampling

In [5]:
eps = 1e-5
spread = (micro['ask'] - micro['bid']).clip(lower=1e-6)
weights = np.sqrt(micro['trade_size'].clip(lower=1.0)) / spread
m = 0.5 * (micro['bid'] + micro['ask'])

micro['canonical_mid'] = m
micro['weight'] = weights
micro['p_clipped'] = micro['canonical_mid'].clip(eps, 1 - eps)

target_event = micro['event_id'].mode().iloc[0]
event_df = micro[micro['event_id'] == target_event].set_index('timestamp').sort_index()

# 1-second grid (change to '100ms' for higher frequency)
resampled = event_df.resample('1s').agg({
    'bid': 'last',
    'ask': 'last',
    'trade_size': 'sum',
    'p_clipped': 'last',
}).ffill()

resampled['spread'] = (resampled['ask'] - resampled['bid']).clip(lower=1e-6)
resampled['canonical_mid'] = 0.5 * (resampled['bid'] + resampled['ask'])
resampled['p_clipped'] = resampled['canonical_mid'].clip(eps, 1 - eps)

resampled.head()

,bid,ask,trade_size,p_clipped,spread,canonical_mid
timestamp,,,,,,
2025-11-14 16:31:16+00:00,0.955,0.99999,0.0,0.977495,0.04499,0.977495


## 4) Save Stage-1 output

This file is consumed by Notebook 2.

In [6]:
out = resampled.reset_index()[['timestamp', 'bid', 'ask', 'trade_size', 'spread', 'p_clipped']]
out.to_csv('stage1_preprocessed.csv', index=False)
print('saved:', Path('stage1_preprocessed.csv').resolve())
print('rows:', len(out))

saved: C:\Users\p\Documents\GitHub\volatility-estimator\research\stage1_preprocessed.csv
rows: 1
